In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import os
import shutil
import pandas as pd
import time

# Load Excel
df = pd.read_excel("odisha_tenders_final_parsed.xlsx")
tid_list = df["TID"].astype(str).tolist()

# Base download dir
base_download_dir = os.path.join(os.getcwd(), "TenderTiger_Downloads")
os.makedirs(base_download_dir, exist_ok=True)

# Chrome setup
options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": base_download_dir,
    "download.prompt_for_download": False,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# Start driver
driver = webdriver.Chrome(options=options)
driver.get("https://www.tendertiger.com/User/Account?login")
input("🔐 Please log in manually, solve CAPTCHA, and press ENTER...")

# Wait for ZIP to finish
def wait_for_download_complete(folder, timeout=30):
    end_time = time.time() + timeout
    while time.time() < end_time:
        if any(f.endswith(".zip") and not f.endswith(".crdownload") for f in os.listdir(folder)):
            return True
        time.sleep(1)
    return False

summary = []

for tid in tid_list:
    print(f"\n🔍 Processing TID: {tid}")
    try:
        driver.get("https://www.tendertiger.com/TenderAI/TenderAIList")
        time.sleep(2)

        # Search by TID
        try:
            search_box = WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.XPATH, "//input[contains(@placeholder,'Search Text')]"))
            )
            search_box.clear()
            search_box.send_keys(tid)

            driver.find_element(By.ID, "btnSearch").click()
        except Exception as e:
            print(f"❌ Search failed for TID: {tid} - {e}")
            summary.append({"TID": tid, "Status": "Search Failed"})
            continue

        # Wait for TID link in results and click it
        try:
            tid_link = WebDriverWait(driver, 15).until(
                EC.element_to_be_clickable((By.XPATH, f"//a[text()='{tid}']"))
            )
            tid_link.click()
        except TimeoutException:
            print(f"❌ TID hyperlink not found or not clickable: {tid}")
            summary.append({"TID": tid, "Status": "TID Link Not Found"})
            continue

        time.sleep(3)

        # Scroll down to load "DownloadAll" link
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        # Click 'DownloadAll'
        try:
            download_all_btn = WebDriverWait(driver, 15).until(
                EC.element_to_be_clickable((By.XPATH, "//a[contains(text(),'DownloadAll')]"))
            )
            download_all_btn.click()
        except TimeoutException:
            print(f"❌ 'DownloadAll' button not found for TID: {tid}")
            summary.append({"TID": tid, "Status": "DownloadAll Not Found"})
            continue

        # Wait for .zip download
        if not wait_for_download_complete(base_download_dir):
            print(f"⏳ Timeout waiting for download: {tid}")
            summary.append({"TID": tid, "Status": "Download Timeout"})
            continue

        # Move to TID folder
        tid_folder = os.path.join(base_download_dir, tid)
        os.makedirs(tid_folder, exist_ok=True)
        for file in os.listdir(base_download_dir):
            if file.endswith(".zip"):
                shutil.move(os.path.join(base_download_dir, file), os.path.join(tid_folder, file))

        print(f"✅ Downloaded for TID: {tid}")
        summary.append({"TID": tid, "Status": "Downloaded"})

    except Exception as e:
        print(f"❌ Unexpected error for TID {tid}: {e}")
        summary.append({"TID": tid, "Status": f"Error: {str(e)}"})

# Save summary log
summary_path = os.path.join(base_download_dir, "TenderTiger_summary.xlsx")
pd.DataFrame(summary).to_excel(summary_path, index=False)
print(f"\n📄 Summary saved at: {summary_path}")

driver.quit()


🔐 Please log in manually, solve CAPTCHA, and press ENTER... 



🔍 Processing TID: 83934997
❌ Search failed for TID: 83934997 - Message: 
Stacktrace:
	GetHandleVerifier [0x0x7ff62ac0fe75+79173]
	GetHandleVerifier [0x0x7ff62ac0fed0+79264]
	(No symbol) [0x0x7ff62a9c9e5a]
	(No symbol) [0x0x7ff62aa20586]
	(No symbol) [0x0x7ff62aa2083c]
	(No symbol) [0x0x7ff62aa74247]
	(No symbol) [0x0x7ff62aa489af]
	(No symbol) [0x0x7ff62aa7100d]
	(No symbol) [0x0x7ff62aa48743]
	(No symbol) [0x0x7ff62aa114c1]
	(No symbol) [0x0x7ff62aa12253]
	GetHandleVerifier [0x0x7ff62aeda2ad+3004797]
	GetHandleVerifier [0x0x7ff62aed46fd+2981325]
	GetHandleVerifier [0x0x7ff62aef3350+3107360]
	GetHandleVerifier [0x0x7ff62ac2a9fe+188622]
	GetHandleVerifier [0x0x7ff62ac3228f+219487]
	GetHandleVerifier [0x0x7ff62ac18dc4+115860]
	GetHandleVerifier [0x0x7ff62ac18f79+116297]
	GetHandleVerifier [0x0x7ff62abff528+11256]
	BaseThreadInitThunk [0x0x7ffc413bdbe7+23]
	RtlUserThreadStart [0x0x7ffc420e5a4c+44]


🔍 Processing TID: 83973694
❌ Search failed for TID: 83973694 - Message: 
Stacktrace:
	Get

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os
import shutil
import pandas as pd
import time

# Load TID list
df = pd.read_excel("odisha_tenders_final_parsed.xlsx")
tid_list = df["TID"].astype(str).tolist()

# Setup download directory
base_download_dir = os.path.join(os.getcwd(), "TenderTiger_Downloads")
os.makedirs(base_download_dir, exist_ok=True)

# Chrome setup
options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": base_download_dir,
    "download.prompt_for_download": False,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
driver.get("https://www.tendertiger.com/User/Account?login")
input("🔐 Please log in manually, solve CAPTCHA, then press ENTER to continue...")

def wait_for_download_complete(folder, timeout=40):
    end_time = time.time() + timeout
    while time.time() < end_time:
        if any(f.endswith(".zip") and not f.endswith(".crdownload") for f in os.listdir(folder)):
            return True
        time.sleep(1)
    return False

summary = []

for tid in tid_list:
    print(f"\n🔍 Processing TID: {tid}")
    try:
        search_url = f"https://www.tendertiger.com/TenderAI/TenderAIList?searchtext={tid}-tenders"
        driver.get(search_url)
        time.sleep(3)

        # Click the tender detail link (with encrypted tenderNo and tcNo)
        try:
            tender_link = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, f"//a[contains(@href,'TenderDetail/Tenderinformation') and contains(@href,'tenderNo=') and contains(@href,'tcNo=')]"))
            )
            tender_url = tender_link.get_attribute("href")
            print(f"➡️ Navigating to tender: {tender_url}")
            driver.get(tender_url)
        except Exception as e:
            print(f"❌ Could not find tender link for TID {tid}: {e}")
            summary.append({"TID": tid, "Status": "Tender Link Not Found"})
            continue

        time.sleep(3)

        # Scroll to bottom for DownloadAll visibility
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        try:
            download_all_btn = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, "//a[contains(text(),'DownloadAll')]"))
            )
            driver.execute_script("arguments[0].click();", download_all_btn)
        except Exception as e:
            print(f"❌ DownloadAll button not found for TID {tid}: {e}")
            summary.append({"TID": tid, "Status": "DownloadAll Not Found"})
            continue

        if not wait_for_download_complete(base_download_dir):
            print(f"⏳ Download timeout for TID: {tid}")
            summary.append({"TID": tid, "Status": "Download Timeout"})
            continue

        tid_folder = os.path.join(base_download_dir, tid)
        os.makedirs(tid_folder, exist_ok=True)
        for f in os.listdir(base_download_dir):
            if f.endswith(".zip"):
                shutil.move(os.path.join(base_download_dir, f), os.path.join(tid_folder, f))

        print(f"✅ Downloaded for TID: {tid}")
        summary.append({"TID": tid, "Status": "Downloaded"})

    except Exception as e:
        screenshot_path = f"{tid}_error.png"
        driver.save_screenshot(screenshot_path)
        print(f"❌ Error for TID {tid}: {e}")
        summary.append({"TID": tid, "Status": f"Error: {str(e) or 'Unknown Error - Screenshot saved at ' + screenshot_path}"})

# Save summary
summary_path = os.path.join(base_download_dir, "TenderTiger_summary.xlsx")
pd.DataFrame(summary).to_excel(summary_path, index=False)
print(f"\n📄 Summary saved at: {summary_path}")

driver.quit()


🔐 Please log in manually, solve CAPTCHA, then press ENTER to continue... 



🔍 Processing TID: 83934997
➡️ Navigating to tender: https://www.tendertiger.com/TenderDetail/Tenderinformation?tenderNo=Kby+6nkjpcdv3mzAeceT8bTbCT0Z6LHyVfAGJB+fQGA=&tcNo=jPhc957nsnMJnEx/qJpoM9it+43j30Wg7XI2iwpivIA=&tendertype=Live
✅ Downloaded for TID: 83934997

🔍 Processing TID: 83973694
➡️ Navigating to tender: https://www.tendertiger.com/TenderDetail/Tenderinformation?tenderNo=khh1gHje/o/b1A7BSfiXMriPjnueF06lvWVZPqsEr90=&tcNo=4BobJfr1TXvB1pe1Ge9/gvaXMFFP0bypOFF/IThWt34=&tendertype=Live
✅ Downloaded for TID: 83973694

🔍 Processing TID: 83927580
➡️ Navigating to tender: https://www.tendertiger.com/TenderDetail/Tenderinformation?tenderNo=J3oPkf5F4FL1ChYo/XnioVgCHbKd9vfLK78BmtbmGzQ=&tcNo=GcDL1II2EdtsVKleMkUE/mTr9XiD+bXn6SbXwBQERTA=&tendertype=Live
✅ Downloaded for TID: 83927580

🔍 Processing TID: 84627731
➡️ Navigating to tender: https://www.tendertiger.com/TenderDetail/Tenderinformation?tenderNo=XvEKOp1Lr960LDfUe9/AeaNG7lXPSTq7Y3VxUifz8FY=&tcNo=aL4sa8anB/s8kuy+pJlIScy3o/wI//VbISQ2zeUj